# Bronze 1: load the raw PDFs

Task `bronze_pdfs` of the job `ingest_financebench`, after `download`. Loads the PDFs in the volume into the Delta
table `bronze_pdfs`, as is. Nothing is parsed or transformed.

```
/Volumes/workspace/finance_bench_bronze/docs/pdfs/*.pdf  -->  workspace.finance_bench_bronze.bronze_pdfs
```

| Column | Type | Holds |
| --- | --- | --- |
| `path` | STRING | The PDF in the volume |
| `modificationTime` | TIMESTAMP | When the file was written |
| `length` | BIGINT | Its size in bytes |
| `content` | BINARY | The raw PDF bytes |

**Incremental.** Auto Loader keeps a checkpoint of the files it has loaded, so a run loads only the new PDFs and a
rerun with nothing new loads 0 rows. No imports are needed: `spark` and `dbutils` come with the notebook.

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
CATALOG = "workspace"
BRONZE_SCHEMA_NAME = "finance_bench_bronze"
BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"

VOLUME = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/docs"
PDFS_DIR = f"{VOLUME}/pdfs"
PDFS_TABLE = f"{BRONZE_SCHEMA}.bronze_pdfs"
CHECKPOINT = f"{VOLUME}/_checkpoints/bronze_pdfs"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.docs")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PDFS_TABLE} (
  path STRING, modificationTime TIMESTAMP, length BIGINT, content BINARY
)""")

## Step 1: load the new PDFs with Auto Loader

Reads every `*.pdf` in `pdfs/` as a stream in the `binaryFile` format, one row per file with its bytes, and appends
the new ones to `bronze_pdfs`. `trigger(availableNow=True)` loads everything new and stops, so it behaves like a batch
task. `awaitTermination()` makes the task wait for the load.

**Example.** The volume holds three PDFs, and an earlier run loaded `3M_2018_10K.pdf`. Sizes are illustrative.

**1a. After `load`.** Auto Loader gives only the files that are not in its checkpoint:

| path | modificationTime | length | content |
| --- | --- | --- | --- |
| dbfs:/Volumes/workspace/finance_bench_bronze/docs/pdfs/PGE_2020_10K.pdf | 2026-11-20 09:14:02 | 4,812,331 | `25 50 44 46 2D 31 2E 36 …` (`%PDF-1.6…`) |
| dbfs:/Volumes/workspace/finance_bench_bronze/docs/pdfs/AMD_2022_10K.pdf | 2026-11-20 09:14:05 | 2,203,918 | `25 50 44 46 2D 31 2E 37 …` |

**1b. After `toTable`, in `bronze_pdfs`.** The 2 new rows are appended to the one already there, the same columns:

| path | modificationTime | length | content |
| --- | --- | --- | --- |
| …/pdfs/3M_2018_10K.pdf | 2026-11-19 17:02:40 | 3,019,442 | `25 50 44 46 …` |
| …/pdfs/PGE_2020_10K.pdf | 2026-11-20 09:14:02 | 4,812,331 | `25 50 44 46 …` |
| …/pdfs/AMD_2022_10K.pdf | 2026-11-20 09:14:05 | 2,203,918 | `25 50 44 46 …` |

A rerun with no new file reads 0 rows and appends nothing.

In [ ]:
(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(PDFS_DIR)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(PDFS_TABLE)
    .awaitTermination()
)

In [ ]:
display(spark.sql(f"SELECT count(*) AS pdfs, round(sum(length) / 1e9, 2) AS gb FROM {PDFS_TABLE}"))